In [8]:
import tensorflow as tf
import numpy as np
class GemmAdd(tf.Module):
    @tf.function(input_signature=[
        tf.TensorSpec([10, 10], tf.float32),
        tf.TensorSpec([10, 10], tf.float32),
        tf.TensorSpec([10, 10], tf.float32),
    ])
    def forward(self, a, b, d):
        return tf.add(tf.linalg.matmul(a, b), d)
        
model = GemmAdd()
cf = model.forward.get_concrete_function()
print(tf.mlir.experimental.convert_function(cf))
tf.saved_model.save(model, "gemm_add_savedmodel",
                    signatures={"serving_default": model.forward})

module attributes {tf.versions = {bad_consumers = [], min_consumer = 0 : i32, producer = 2474 : i32}} {
  func.func @__inference_forward_88(%arg0: tensor<10x10xf32> {tf._user_specified_name = "a"}, %arg1: tensor<10x10xf32> {tf._user_specified_name = "b"}, %arg2: tensor<10x10xf32> {tf._user_specified_name = "d"}) -> tensor<10x10xf32> attributes {allow_soft_placement = false, tf.entry_function = {control_outputs = "", inputs = "a,b,d", outputs = "identity_RetVal"}} {
    %0 = "tf.MatMul"(%arg0, %arg1) <{grad_a = false, grad_b = false, transpose_a = false, transpose_b = false}> {device = ""} : (tensor<10x10xf32>, tensor<10x10xf32>) -> tensor<10x10xf32>
    %1 = "tf.AddV2"(%0, %arg2) {device = ""} : (tensor<10x10xf32>, tensor<10x10xf32>) -> tensor<10x10xf32>
    %2 = "tf.Identity"(%1) {device = ""} : (tensor<10x10xf32>) -> tensor<10x10xf32>
    return %2 : tensor<10x10xf32>
  }
}

INFO:tensorflow:Assets written to: gemm_add_savedmodel/assets


INFO:tensorflow:Assets written to: gemm_add_savedmodel/assets


# Tensorflow lowering path to rocMLIR

The Tensorflow model from above is using the TF Dialect. See below for the TF dialect

```
module attributes {tf.versions = {bad_consumers = [], min_consumer = 0 : i32, producer = 2474 : i32}} {
  func.func @__inference_forward_88(%arg0: tensor<10x10xf32> {tf._user_specified_name = "a"}, %arg1: tensor<10x10xf32> {tf._user_specified_name = "b"}, %arg2: tensor<10x10xf32> {tf._user_specified_name = "d"}) -> tensor<10x10xf32> attributes {allow_soft_placement = false, tf.entry_function = {control_outputs = "", inputs = "a,b,d", outputs = "identity_RetVal"}} {
    %0 = "tf.MatMul"(%arg0, %arg1) <{grad_a = false, grad_b = false, transpose_a = false, transpose_b = false}> {device = ""} : (tensor<10x10xf32>, tensor<10x10xf32>) -> tensor<10x10xf32>
    %1 = "tf.AddV2"(%0, %arg2) {device = ""} : (tensor<10x10xf32>, tensor<10x10xf32>) -> tensor<10x10xf32>
    %2 = "tf.Identity"(%1) {device = ""} : (tensor<10x10xf32>) -> tensor<10x10xf32>
    return %2 : tensor<10x10xf32>
  }
}
```

Running the following command to go from tensorflow to stablehlo (note that the stablehlo is in byte code format)

```
iree-import-tf --tf-import-type=savedmodel_v2  \
    --tf-savedmodel-exported-names=forward   \
    gemm_add_savedmodel -o gemm_add_stablehlo.mlir
```

We can see the following IR in stablehlo: 

```
module {
  func.func @forward(%arg0: tensor<10x10xf32>, %arg1: tensor<10x10xf32>, %arg2: tensor<10x10xf32>) -> tensor<10x10xf32> {
    %0 = stablehlo.dot %arg0, %arg1, precision = [DEFAULT, DEFAULT] : (tensor<10x10xf32>, tensor<10x10xf32>) -> tensor<10x10xf32>
    %1 = stablehlo.add %0, %arg2 : tensor<10x10xf32>
    return %1 : tensor<10x10xf32>
  }
}

```

Running the following command to go from stablehlo into linalg

```
~/frameworks/stablehlo/build/bin/stablehlo-opt gemm_add_stablehlo.mlir --stablehlo-legalize-to-linalg=enable-primitive-ops
```

Output: 

```
module {
  func.func @forward(%arg0: tensor<10x10xf32>, %arg1: tensor<10x10xf32>, %arg2: tensor<10x10xf32>) -> tensor<10x10xf32> {
    %0 = tensor.empty() : tensor<10x10xf32>
    %cst = arith.constant 0.000000e+00 : f32
    %1 = linalg.fill ins(%cst : f32) outs(%0 : tensor<10x10xf32>) -> tensor<10x10xf32>
    %2 = linalg.matmul ins(%arg0, %arg1 : tensor<10x10xf32>, tensor<10x10xf32>) outs(%1 : tensor<10x10xf32>) -> tensor<10x10xf32>
    %3 = tensor.empty() : tensor<10x10xf32>
    %mapped = linalg.map { arith.addf } ins(%2, %arg2 : tensor<10x10xf32>, tensor<10x10xf32>) outs(%3 : tensor<10x10xf32>)
    return %mapped : tensor<10x10xf32>
  }
}
```

Finally, outputting rocMLIR from linalg:

```
~/rocMLIR/build-release/bin/rocmlir-gen rocmlir.mlir --clone-harness --fut forward --arch gfx950  | ~/rocMLIR/build-release/bin/rocmlir-driver --kernel-pipeline=highlevel
```

We have the following code

```
#map = affine_map<(d0, d1) -> (d0, d1)>
#map1 = affine_map<(d0, d1, d2) -> (d0, d2)>
#map2 = affine_map<(d0, d1, d2) -> (d2, d1)>
#map3 = affine_map<(d0, d1, d2) -> (d0, d1)>
module {
  func.func @forward(%arg0: memref<10x10xf32> {mhal.read_access}, %arg1: memref<10x10xf32> {mhal.read_access}, %arg2: memref<10x10xf32> {mhal.read_access}, %arg3: memref<10x10xf32> {mhal.write_access}) {
    %cst = arith.constant 0.000000e+00 : f32
    %alloc = memref.alloc() {alignment = 64 : i64} : memref<10x10xf32>
    linalg.generic {indexing_maps = [#map], iterator_types = ["parallel", "parallel"]} outs(%alloc : memref<10x10xf32>) {
    ^bb0(%out: f32):
      linalg.yield %cst : f32
    }
    linalg.generic {indexing_maps = [#map1, #map2, #map3], iterator_types = ["parallel", "parallel", "reduction"]} ins(%arg0, %arg1 : memref<10x10xf32>, memref<10x10xf32>) outs(%alloc : memref<10x10xf32>) {
    ^bb0(%in: f32, %in_1: f32, %out: f32):
      %0 = arith.mulf %in, %in_1 : f32
      %1 = arith.addf %out, %0 : f32
      linalg.yield %1 : f32
    }
    %alloc_0 = memref.alloc() {alignment = 64 : i64} : memref<10x10xf32>
    linalg.generic {indexing_maps = [#map, #map, #map], iterator_types = ["parallel", "parallel"]} ins(%alloc, %arg2 : memref<10x10xf32>, memref<10x10xf32>) outs(%alloc_0 : memref<10x10xf32>) {
    ^bb0(%in: f32, %in_1: f32, %out: f32):
      %0 = arith.addf %in, %in_1 : f32
      linalg.yield %0 : f32
    }
    memref.copy %alloc_0, %arg3 : memref<10x10xf32> to memref<10x10xf32>
    return
  }
  func.func @forward_wrapper(%arg0: memref<10x10xf32>, %arg1: memref<10x10xf32>, %arg2: memref<10x10xf32>, %arg3: memref<10x10xf32>) {
    %alloc = memref.alloc() : memref<10x10xf32>
    %token = mhal.launch @forward (%arg0, %arg1, %arg2, %alloc) : (memref<10x10xf32>, memref<10x10xf32>, memref<10x10xf32>, memref<10x10xf32>)
    mhal.await %token : !mhal.token
    memref.copy %alloc, %arg3 : memref<10x10xf32> to memref<10x10xf32>
    return
  }
  module @__xmodule_ attributes {mhal.arch = "gfx950", mhal.module} {
    func.func @forward(%arg0: memref<10x10xf32> {mhal.read_access}, %arg1: memref<10x10xf32> {mhal.read_access}, %arg2: memref<10x10xf32> {mhal.read_access}, %arg3: memref<10x10xf32> {mhal.write_access}) attributes {kernel, original_func = @forward} {
      %alloc = memref.alloc() {alignment = 64 : i64} : memref<10x10xf32>
      rock.gemm %alloc = %arg0 * %arg1 storeMethod =  set : memref<10x10xf32> = memref<10x10xf32> * memref<10x10xf32>
      %alloc_0 = memref.alloc() {alignment = 64 : i64} : memref<10x10xf32>
      linalg.generic {indexing_maps = [#map, #map, #map], iterator_types = ["parallel", "parallel"]} ins(%alloc, %arg2 : memref<10x10xf32>, memref<10x10xf32>) outs(%alloc_0 : memref<10x10xf32>) {
      ^bb0(%in: f32, %in_1: f32, %out: f32):
        %0 = arith.addf %in, %in_1 : f32
        linalg.yield %0 : f32
      }
      memref.copy %alloc_0, %arg3 : memref<10x10xf32> to memref<10x10xf32>
      return
    }
  }
}
```